# Limpieza de grado de ocupación hotelera — TFG (versión Colab)
**Beatriz Muñoz García-Serrano · Grado en Business Analytics · UFV**

## Archivos que debes tener en `/content/` antes de ejecutar
| Archivo | Descripción |
|---|---|
| `2 grado ocup plaza.csv` | Ocupación general mensual (INE, sep=';', latin1) |
| `2 grado ocupación por plazas finde semana.csv` | Ocupación fin de semana (INE, sep=';', latin1) |

## Output generado
- `/content/2 hoteles_clean.csv` — archivo que usa directamente `integracion_datasets_COLAB.ipynb`

**Nota:** los reads sin ruta explícita (`pd.read_csv('archivo.csv')`) funcionan en Colab
porque el directorio de trabajo es `/content/`. Solo se ha cambiado el nombre del output
para que coincida con el que espera `integracion_datasets_COLAB.ipynb`.


In [ ]:
import pandas as pd

In [ ]:
general = pd.read_csv("2 grado ocup plaza.csv", sep=";", encoding="latin1")
finde = pd.read_csv("2 grado ocupación por plazas finde semana.csv", sep=";", encoding="latin1")

In [ ]:
print("GENERAL:", general.shape)
display(general.head())

print("FINDE:", finde.shape)
display(finde.head())

GENERAL: (882, 4)


,Puntos turísticos,Establecimientos y personal empleado (plazas),Periodo,Total
0,18134 Monachil,Grado de ocupación por plazas,2026M02,"54,40"
1,18134 Monachil,Grado de ocupación por plazas,2026M01,"62,36"
2,18134 Monachil,Grado de ocupación por plazas,2025M12,"45,11"
3,18134 Monachil,Grado de ocupación por plazas,2025M11,.
4,18134 Monachil,Grado de ocupación por plazas,2025M10,.


FINDE: (882, 4)


,Puntos turísticos,Establecimientos y personal empleado (plazas),Periodo,Total
0,18134 Monachil,Grado de ocupación por plazas en fin de semana,2026M02,"55,05"
1,18134 Monachil,Grado de ocupación por plazas en fin de semana,2026M01,"68,85"
2,18134 Monachil,Grado de ocupación por plazas en fin de semana,2025M12,"53,96"
3,18134 Monachil,Grado de ocupación por plazas en fin de semana,2025M11,.
4,18134 Monachil,Grado de ocupación por plazas en fin de semana,2025M10,.


In [ ]:
general = general.rename(columns={
    "Puntos turísticos": "puntos_turisticos",
    "Establecimientos y personal empleado (plazas)": "indicador",
    "Periodo": "periodo",
    "Total": "ocupacion_pct"
})

finde = finde.rename(columns={
    "Puntos turísticos": "puntos_turisticos",
    "Establecimientos y personal empleado (plazas)": "indicador",
    "Periodo": "periodo",
    "Total": "ocupacion_pct"
})

In [ ]:
general["tipo_periodo"] = "general"
finde["tipo_periodo"] = "fin_de_semana"

In [ ]:
hoteles = pd.concat([general, finde], ignore_index=True)

print(hoteles.shape)
display(hoteles.head())

(1764, 5)


,puntos_turisticos,indicador,periodo,ocupacion_pct,tipo_periodo
0,18134 Monachil,Grado de ocupación por plazas,2026M02,"54,40",general
1,18134 Monachil,Grado de ocupación por plazas,2026M01,"62,36",general
2,18134 Monachil,Grado de ocupación por plazas,2025M12,"45,11",general
3,18134 Monachil,Grado de ocupación por plazas,2025M11,.,general
4,18134 Monachil,Grado de ocupación por plazas,2025M10,.,general


In [ ]:
hoteles["ocupacion_pct"] = (
    hoteles["ocupacion_pct"]
    .astype(str)
    .str.strip()
    .replace(".", pd.NA)
    .str.replace(",", ".", regex=False)
)

hoteles["ocupacion_pct"] = pd.to_numeric(hoteles["ocupacion_pct"], errors="coerce")

In [ ]:
hoteles[["periodo", "tipo_periodo", "ocupacion_pct"]].head(10)

,periodo,tipo_periodo,ocupacion_pct
0,2026M02,general,54.40
1,2026M01,general,62.36
2,2025M12,general,45.11
3,2025M11,general,NaN
4,2025M10,general,NaN
5,2025M09,general,NaN
6,2025M08,general,NaN
7,2025M07,general,NaN
8,2025M06,general,NaN
9,2025M05,general,NaN


In [ ]:
hoteles["puntos_turisticos"] = hoteles["puntos_turisticos"].astype(str).str.strip()

hoteles["codigo_punto_turistico"] = hoteles["puntos_turisticos"].str.extract(r"^(\d+)")
hoteles["punto_turistico"] = hoteles["puntos_turisticos"].str.extract(r"^\d+\s+(.*)")

In [ ]:
hoteles[["puntos_turisticos", "codigo_punto_turistico", "punto_turistico"]].head()

,puntos_turisticos,codigo_punto_turistico,punto_turistico
0,18134 Monachil,18134,Monachil
1,18134 Monachil,18134,Monachil
2,18134 Monachil,18134,Monachil
3,18134 Monachil,18134,Monachil
4,18134 Monachil,18134,Monachil


In [ ]:
hoteles["anio"] = hoteles["periodo"].str.extract(r"^(\d{4})").astype("Int64")
hoteles["mes"] = hoteles["periodo"].str.extract(r"M(\d{2})$").astype("Int64")

In [ ]:
hoteles["fecha_mes_ref"] = pd.to_datetime(
    hoteles["anio"].astype(str) + "-" + hoteles["mes"].astype(str).str.zfill(2) + "-01",
    errors="coerce"
)

In [ ]:
hoteles[["periodo", "anio", "mes", "fecha_mes_ref"]].head()

,periodo,anio,mes,fecha_mes_ref
0,2026M02,2026,2,2026-02-01
1,2026M01,2026,1,2026-01-01
2,2025M12,2025,12,2025-12-01
3,2025M11,2025,11,2025-11-01
4,2025M10,2025,10,2025-10-01


In [ ]:
hoteles["tipo_periodo"] = hoteles["tipo_periodo"].astype(str).str.strip().str.lower()
hoteles["punto_turistico"] = hoteles["punto_turistico"].astype(str).str.strip()
hoteles["codigo_punto_turistico"] = pd.to_numeric(hoteles["codigo_punto_turistico"], errors="coerce")

In [ ]:
columnas_finales = [
    "codigo_punto_turistico",
    "punto_turistico",
    "indicador",
    "tipo_periodo",
    "anio",
    "mes",
    "fecha_mes_ref",
    "periodo",
    "ocupacion_pct"
]

hoteles_clean_full = hoteles[columnas_finales].copy()

In [ ]:
hoteles_clean_full = hoteles_clean_full.sort_values(
    by=["punto_turistico", "tipo_periodo","fecha_mes_ref" , "anio", "mes"]
).reset_index(drop=True)

display(hoteles_clean_full.head(20))

,codigo_punto_turistico,punto_turistico,indicador,tipo_periodo,anio,mes,fecha_mes_ref,periodo,ocupacion_pct
0,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,1,2018-01-01,2018M01,60.14
1,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,2,2018-02-01,2018M02,88.49
2,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,3,2018-03-01,2018M03,71.29
3,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,4,2018-04-01,2018M04,35.93
4,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,5,2018-05-01,2018M05,NaN
5,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,6,2018-06-01,2018M06,54.50
6,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,7,2018-07-01,2018M07,64.68
7,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,8,2018-08-01,2018M08,68.58
8,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,9,2018-09-01,2018M09,49.63
9,22054,Benasque,Grado de ocupación por plazas en fin de semana,fin_de_semana,2018,10,2018-10-01,2018M10,55.90


In [ ]:
hoteles_clean_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1764 entries, 0 to 1763
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   codigo_punto_turistico  1764 non-null   int64         
 1   punto_turistico         1764 non-null   object        
 2   indicador               1764 non-null   object        
 3   tipo_periodo            1764 non-null   object        
 4   anio                    1764 non-null   Int64         
 5   mes                     1764 non-null   Int64         
 6   fecha_mes_ref           1764 non-null   datetime64[ns]
 7   periodo                 1764 non-null   object        
 8   ocupacion_pct           1066 non-null   float64       
dtypes: Int64(2), datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 127.6+ KB


In [ ]:
hoteles_clean_full.isnull().sum().sort_values(ascending=False)

,0
ocupacion_pct,698
codigo_punto_turistico,0
punto_turistico,0
tipo_periodo,0
indicador,0
anio,0
mes,0
fecha_mes_ref,0
periodo,0


In [ ]:
hoteles_clean_full.to_csv("/content/2 hoteles_clean.csv", index=False, encoding="utf-8-sig", sep=";")

In [ ]:
from google.colab import files
files.download("/content/2 hoteles_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>